In [ ]:
import numpy as np 
from sklearn.datasets import make_blobs
from sklearn.preprocessing import MinMaxScaler

from bucket import create_bucket_synopsis, Params
from lloyd import lloyd_with_weights
from evaluation_utils import kmeans_loss

master_rng = np.random.default_rng(42)

def lsh_experiment(data: np.ndarray, p: Params, gaussian: bool = False, n_trials: int = 20):
    s = master_rng.integers(low=0, high=100000)
    total_loss = 0
    n_successful_trials = n_trials
    for x in range(n_trials):
        if gaussian:
            private_points, private_weights = create_bucket_synopsis(data, p, s+x, use_gaussian=True)
        else:
            private_points, private_weights = create_bucket_synopsis(data, p, s+x)
        if private_points.shape[0] <= p.k: # if number of points is less than or equal to desired number of centers
            centers = private_points
        else:
            centers = lloyd_with_weights(k=p.k, X=private_points, weights=private_weights, n_iter=5, rs=s+x)
        try:
            loss = kmeans_loss(centers, data)
        except:
            loss = 0
            n_successful_trials -=1
        total_loss += loss
        print(f"Trial {x+1} done")
    print("Number completed trials: ", n_successful_trials)
    return total_loss / n_successful_trials

# d - dimension
# k - number of clusters
# n - number of points
def make_dataset(d: int, k: int, n: int=10000):
    data, _ = make_blobs(n_samples=n, n_features=d, centers=k)
    print(type(data))
    # now normalise it to (-1,1) meaning a radius of 1.4
    scaler = MinMaxScaler((-1,1))
    normalised_data = scaler.fit_transform(data)
    return normalised_data

In [ ]:
dimensions = range(10, 501, 10)
r_gaussian = []
r_laplace = []
for d in dimensions:
    data = make_dataset(d, 10, 10000)
    p = Params(epsilon=1, delta=1e-6, radius=np.sqrt(d), dimension=d, k=10, max_depth=15)
    r_gaussian.append(lsh_experiment(data, p, True))
    r_laplace.append(lsh_experiment(data, p, False))